<a href="https://colab.research.google.com/github/amanouz-mounir/Classification-of-Aviation-Incident-reports/blob/main/Predict_with_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)

In [ ]:
def bert_text_preparation(text, tokenizer):
    """Preparing the input for BERT

    Takes a string argument and performs
    pre-processing like adding special tokens,
    tokenization, tokens to ids, and tokens to
    segment ids. All tokens are mapped to seg-
    ment id = 1.

    Args:
        text (str): Text to be converted
        tokenizer (obj): Tokenizer object
            to convert text into BERT-re-
            adable tokens and ids

    Returns:
        list: List of BERT-readable tokens
        obj: Torch tensor with token ids
        obj: Torch tensor segment ids


    """
    marked_text = "[CLS] " + text + " [SEP]"
    tokenized_text = tokenizer.tokenize(marked_text)
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    segments_ids = [1]*len(indexed_tokens)

    # Convert inputs to PyTorch tensors
    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segments_ids])

    return tokenized_text, tokens_tensor, segments_tensors


In [ ]:
def get_bert_sentence_embedding(tokens_tensor, segments_tensors, model, max_length=512):
    device = tokens_tensor.device  # Ensure all tensors are on the same device

    all_token_embeddings = []

    # Ensure tensors have correct shape
    if tokens_tensor.dim() == 1:
        tokens_tensor = tokens_tensor.unsqueeze(0)
    if segments_tensors.dim() == 1:
        segments_tensors = segments_tensors.unsqueeze(0)

    if tokens_tensor.size(1) > max_length:
        num_chunks = (tokens_tensor.size(1) + max_length - 1) // max_length

        for i in range(num_chunks):
            start_idx = i * max_length
            end_idx = min((i + 1) * max_length, tokens_tensor.size(1))

            chunk_tokens = tokens_tensor[:, start_idx:end_idx]
            chunk_segments = segments_tensors[:, start_idx:end_idx]

            chunk_tokens = chunk_tokens.to(device)
            chunk_segments = chunk_segments.to(device)

            with torch.no_grad():
                outputs = model(chunk_tokens, token_type_ids=chunk_segments, output_hidden_states=True)
                hidden_states = outputs.hidden_states[-1]
                token_embeddings = hidden_states.squeeze(0)

            all_token_embeddings.append(token_embeddings.mean(dim=0))

        sentence_embedding = torch.mean(torch.stack(all_token_embeddings), dim=0)

    else:
        tokens_tensor = tokens_tensor.to(device)
        segments_tensors = segments_tensors.to(device)

        with torch.no_grad():
            outputs = model(tokens_tensor, token_type_ids=segments_tensors, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]
            token_embeddings = hidden_states.squeeze(0)

        sentence_embedding = torch.mean(token_embeddings, dim=0)

    return sentence_embedding.cpu().numpy()  # Move the result back to CPU before converting to numpy

In [ ]:
def embedding(text) :
  tokenized_text, tokens_tensor, segments_tensors = bert_text_preparation(text, tokenizer)
  sentence_embedding = get_bert_sentence_embedding(tokens_tensor, segments_tensors, bert_model)
  return sentence_embedding

In [ ]:
import pandas as pd
df_test = pd.read_csv('/content/Dataset_final_for_LLM_test.csv')


In [ ]:
import joblib

xgb_model = joblib.load("/content/xgb_model_multilabel_best.joblib")
lgbm_model = joblib.load("/content/lgbm_model_multilabel_prob.joblib")


In [ ]:
import numpy as np
import shap

# Pipeline pour SHAP : texte → BERT → XGBoost
def shap_predict(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()

    embeddings = [embedding(text) for text in texts]
    X = np.vstack(embeddings)
    return xgb_model.predict_proba(X)  # shape (n_samples, n_classes)

# 5. Texte à expliquer
text = "AN UNUSUAL ODOR WAS note IN THE COCKPIT BY flightcrew.A FEW minute LATER..."
print(shap_predict([text]))
# 6. SHAP : texte → masking → prédiction
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(shap_predict, masker)

# 7. Calcul des valeurs SHAP
shap_values = explainer([text], fixed_context=1)

# 8. Affichage
shap.plots.text(shap_values[0])

[[0.29590085 0.2852289  0.12844197 0.11041887 0.23074916 0.35798883
  0.11086497 0.11028218 0.1334054  0.1660517  0.2076113  0.1381448
  0.31750754 0.0966241  0.24547444 0.15445116 0.23513459 0.18726382
  0.3134604  0.24547806 0.29793015 0.32402146]]
